In [0]:
# ============================================================
# Batch ETL demo: landing -> bronze -> silver -> gold
# Storage: azure4helen / lakehouse  (keyless via Unity Catalog)
# ============================================================

base = "abfss://lakehouse@azure4helen.dfs.core.windows.net"
landing       = f"{base}/landing"
bronze_orders = f"{base}/bronze/orders_al"     # where Auto Loader writes
silver_orders = f"{base}/silver/orders"
gold_revenue  = f"{base}/gold/revenue_by_channel"

# Unique filename each run = simulates a genuinely new file drop
from datetime import datetime
fname = f"orders_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

sample_csv = """order_id,partner,channel,amount,order_date
1001,Acme,WHS,250.00,2026-06-01
1002,Acme,PP,99.50,2026-06-01
1003,Globex,CR,1200.00,2026-06-02
1004,Globex,WHS,,2026-06-02
1005,Initech,PP,45.00,2026-06-03
1003,Globex,CR,1200.00,2026-06-02
"""
dbutils.fs.put(f"{landing}/{fname}", sample_csv, overwrite=True)
print(f"Sample file written to landing/{fname}")

In [0]:
from pyspark.sql.functions import current_timestamp, col

df_stream = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_loc)
    .option("pathGlobFilter", "orders*.csv")
    .option("header", "true")
    .load(landing))

df_bronze = (df_stream
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", col("_metadata.file_path")))   # <- UC-friendly

(df_bronze.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint)
    .trigger(availableNow=True)
    .start(bronze)
    .awaitTermination())

print("Auto Loader run complete.")
display(spark.read.format("delta").load(bronze))

In [0]:
# ------------------------------------------------------------
# 1. BRONZE: read raw, add metadata, write as Delta (faithful capture)
# ------------------------------------------------------------
from pyspark.sql.functions import current_timestamp, input_file_name, lit

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(f"{landing}/orders.csv"))

df_bronze = (df_raw
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("orders.csv")))

df_bronze.write.format("delta").mode("overwrite").save(f"{bronze}/orders")
print("Bronze written.")
display(spark.read.format("delta").load(f"{bronze}/orders"))

In [0]:
from pyspark.sql.functions import col, to_date

df_silver = (spark.read.format("delta").load(bronze_orders)
    .dropDuplicates(["order_id", "partner", "channel", "order_date"])
    .filter(col("amount").isNotNull())
    .withColumn("order_id",   col("order_id").cast("string"))    # pin it
    .withColumn("amount",     col("amount").cast("double"))
    .withColumn("order_date", to_date(col("order_date"))))

df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(silver_orders)

print("Silver written.")
display(spark.read.format("delta").load(silver_orders))

In [0]:
# ------------------------------------------------------------
# 3. GOLD: business aggregate — revenue per channel
# ------------------------------------------------------------
from pyspark.sql.functions import (
    col, lit, current_timestamp, input_file_name,
    to_date, sum as _sum, count
)

df_gold = (spark.read.format("delta").load(f"{silver}/orders")
    .groupBy("channel")
    .agg(_sum("amount").alias("total_revenue"),
         count("*").alias("order_count"))
    .orderBy(col("total_revenue").desc()))

df_gold.write.format("delta").mode("overwrite").save(f"{gold}/revenue_by_channel")
print("Gold written.")
display(spark.read.format("delta").load(f"{gold}/revenue_by_channel"))